# population

In [ ]:
#| default_exp game/flag

In [ ]:
#| export
from fastcore.basics import patch

# fun with colors
import colorsys
import seaborn as sns
import matplotlib.pyplot as plt
from importlib import resources
import pandas as pd
import random
import re
from enum import Enum

In [ ]:
#| export
from HexMagic.styles import   SVGBuilder, SVGDef,  Generatable, NamedColor, StyleCSS
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord , Hex, HexGrid, PrimitiveDemo, HexWrapper, HexTouchMap
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion ,  unique_windy_edge
from HexMagic.terrainpatterns import TerrainPatterns, PathPattern
from HexMagic.styles import apply_looping_animation, LoopingLayerAnimation
from HexMagic.terrain import Terrain


import chess.svg

In [ ]:
#| export
class PieceType(Enum):
    """Chess piece types."""
    KING   = "king"
    QUEEN  = "queen"
    BISHOP = "bishop"
    KNIGHT = "knight"
    ROOK   = "rook"
    PAWN   = "pawn"

    @property
    def icon(self):
        return {"king": "♚", "queen": "♛", "bishop": "♝",
                "knight": "♞", "rook": "♜", "pawn": "♟"}[self.value]


In [ ]:
#| export
class CountryFlag:

    def __init__(self, c, name, year,patternIndex = 0):
        #self.primary = plt.matplotlib.colors.rgb2hex(c)
        h, s, v = colorsys.rgb_to_hsv(*c)
        
        self.primary = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb(h, s * 0.4, min(v * 1.4, 1.0)))
        self.darkPrimary = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb(h, s, max(v * 0.35, 0.05)))
        self.comp = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h + 0.5) % 1, s, max(v * 0.35, 0.05)))

        self.basePrimary =  colorsys.rgb_to_hsv(*c)
        self.baseComp = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h + 0.5) % 1, s, v))
        self.tri1 = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h + 1/3) % 1, s, v))
        self.tri2 = plt.matplotlib.colors.rgb2hex(colorsys.hsv_to_rgb((h - 1/3) % 1, s, v))
        self.name = name
        self.year = year
        self.countryPrefix = CountryFlag.countryName(name)
        self.capital = CountryFlag.cityName(name)
        self.patternIndex = patternIndex

    @classmethod
    def seaborn(cls,name:str,levels=7,gender=None,year=1900):
        palette = sns.color_palette(name, levels)

        if gender is not None:
            names = random.sample(CountryFlag.commonNames(year=year,gender=gender),levels)
        else:
            allN = CountryFlag.commonNames(year=year,gender="M") + CountryFlag.commonNames(year=year,gender="F")
            names = random.sample(allN,levels)
            
       
        ret = []
        for i, color in enumerate(palette):
            ret.append(cls(color,name=names[i],year=year))
        return ret

    
    @classmethod
    def commonNames(cls,year=1900,gender="M"):
        
        with resources.files('HexMagic').joinpath('data/misc/popularNames.csv').open() as f:
            df = pd.read_csv(f)
        
        filtered = df[(df['Year'] == year) & (df['Gender'] == gender)]
        if filtered.empty:
            raise ValueError(f"No names found for year={year}, gender={gender}")
        
        # Weight by count for more realistic distribution
        return filtered['Name'].tolist()


    @classmethod
    def countryName(cls, name , pattern=None, descriptor=None):
        # Dictionary of place descriptors organized by first letter
        PLACE_DESCRIPTORS = {
            'A': ['Abbey', 'Acres', 'Alcove', 'Apex', 'Archipelago', 'Arena', 'Atoll', 'Avenue'],
            'B': ['Basin', 'Bay', 'Bluff', 'Borough', 'Boundary', 'Bower', 'Burg', 'Borderlands'],
            'C': ['Canyon', 'Cape', 'Castle', 'Citadel', 'Clearing', 'Cove', 'Crossing', 'County'],
            'D': ['Dale', 'Dell', 'Delta', 'Den', 'District', 'Domain', 'Dunes', 'Dominion'],
            'E': ['Edge', 'Enclave', 'End', 'Estate', 'Expanse', 'Empire', 'Escarpment', 'Eyrie'],
            'F': ['Falls', 'Fen', 'Fjord', 'Forest', 'Fort', 'Frontier', 'Fields', 'Fief'],
            'G': ['Gap', 'Garden', 'Gate', 'Glade', 'Glen', 'Gorge', 'Grotto', 'Grove'],
            'H': ['Habitat', 'Harbor', 'Haven', 'Heath', 'Heights', 'Hideaway', 'Hill', 'Hollow'],
            'I': ['Isle', 'Inlet', 'Island', 'Isthmus', 'Ironworks', 'Imperium', 'Inn', 'Impasse'],
            'J': ['Junction', 'Jungle', 'Jetty', 'Juncture', 'Jurisdiction', 'Jut', 'Joint', 'Jewel'],
            'K': ['Keep', 'Kingdom', 'Knoll', 'Key', 'Knot', 'Kiosk', 'Krantz', 'Karst'],
            'L': ['Lagoon', 'Lake', 'Landing', 'Land', 'Lair', 'Ledge', 'Lodge', 'Lowlands'],
            'M': ['Manor', 'Marsh', 'Meadow', 'Mesa', 'Moor', 'Mount', 'Mountains', 'Mound'],
            'N': ['Narrows', 'Nest', 'Niche', 'Nook', 'North', 'Notch', 'Nation', 'Neighborhood'],
            'O': ['Oasis', 'Observatory', 'Outpost', 'Overlook', 'Orchard', 'Outcrop', 'Outlet', 'Outlands'],
            'P': ['Palace', 'Park', 'Pass', 'Path', 'Peak', 'Peninsula', 'Pinnacle', 'Plaza', 'Point', 'Province'],
            'Q': ['Quarry', 'Quarter', 'Quay', 'Quarters', 'Quad', 'Quadrant', 'Quest', 'Quietude'],
            'R': ['Range', 'Ravine', 'Reach', 'Realm', 'Reef', 'Region', 'Reserve', 'Retreat', 'Ridge', 'Rise'],
            'S': ['Sanctuary', 'Settlement', 'Shire', 'Shore', 'Slopes', 'Sound', 'Span', 'Spring', 'Summit', 'Stronghold'],
            'T': ['Terrace', 'Territory', 'Thicket', 'Timberland', 'Tower', 'Town', 'Trail', 'Trench', 'Tundra', 'Township'],
            'U': ['Undergrowth', 'Underpass', 'Union', 'Uplands', 'Upper', 'Utopia', 'Utterness', 'Umbrage'],
            'V': ['Vale', 'Valley', 'Vault', 'View', 'Villa', 'Village', 'Vineyards', 'Vista', 'Void', 'Vanguard'],
            'W': ['Ward', 'Wasteland', 'Water', 'Way', 'Wetlands', 'Wilds', 'Wood', 'Woods', 'Works', 'Warren'],
            'X': ['Xanadu', 'Xenolith', 'Xerophyte', 'X-Roads', 'Xeric', 'Xyst', 'X-Point', 'X-ing'],
            'Y': ['Yard', 'Yonder', 'Yurt', 'Yards', 'Yielding', 'York', 'Yukon', 'Yews'],
            'Z': ['Zone', 'Zenith', 'Zephyr', 'Zigzag', 'Ziggurat', 'Zion', 'Zodiac', 'Zocalo'],
        }

        # Possessive patterns
        PATTERNS = [
            "{name}'s {place}",      # Karl's Kingdom
            "{place} of {name}",     # Kingdom of Karl
            "{name} {place}",        # Karl Kingdom
        ]

        if not name:
            return ""
        
        first_letter = name[0].upper()
        
        # Get possible descriptors for this letter
        descriptors = PLACE_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Precinct'])
        
        # Choose descriptor
        if descriptor and descriptor in descriptors:
            place = descriptor
        else:
            place = random.choice(descriptors)

        return place
        
        # Choose pattern
        if pattern is not None and 0 <= pattern < len(PATTERNS):
            template = PATTERNS[pattern]
        else:
            template = random.choice(PATTERNS)
        
        return template.format(name=name, place=place)

    @classmethod
    def cityName(cls, name , pattern=None, descriptor=None, use_suffix=None):
        # Dictionary of place descriptors organized by first letter
        SETTLEMENT_DESCRIPTORS = {
            'A': ['Acres', 'Arbor', 'Ashton', 'Auburn', 'Avon', 'Aldridge', 'Ashford', 'Aston'],
            'B': ['Bay', 'Beach', 'Bridge', 'Brook', 'Burg', 'Borough', 'Bluff', 'Bend'],
            'C': ['City', 'Cove', 'Creek', 'Crest', 'Crossing', 'Center', 'Cape', 'Corners'],
            'D': ['Dale', 'Dell', 'Dunes', 'Down', 'Dock', 'Delta', 'Downs', 'Den'],
            'E': ['End', 'Edge', 'Estates', 'Elms', 'Enclave', 'Evergreen', 'East', 'Elm'],
            'F': ['Falls', 'Field', 'Fields', 'Ford', 'Forest', 'Fort', 'Forks', 'Ferry'],
            'G': ['Glen', 'Glade', 'Green', 'Grove', 'Gate', 'Gardens', 'Groves', 'Gap'],
            'H': ['Harbor', 'Haven', 'Heights', 'Hill', 'Hills', 'Hollow', 'Heath', 'Hurst'],
            'I': ['Isle', 'Island', 'Inlet', 'Inn', 'Ironworks', 'Ivy', 'Isles', 'Inches'],
            'J': ['Junction', 'Jetty', 'Juncture', 'Junction', 'Jamestown', 'Jardin', 'Jct', 'Joya'],
            'K': ['Key', 'Knoll', 'Knolls', 'Keep', 'Keystone', 'Kingswood', 'Kirk', 'Knolle'],
            'L': ['Lake', 'Landing', 'Lawn', 'Ledge', 'Lock', 'Lodge', 'Lagoon', 'Lynn'],
            'M': ['Manor', 'Meadow', 'Meadows', 'Mill', 'Mills', 'Mount', 'Moor', 'Mountain'],
            'N': ['North', 'Nook', 'Narrows', 'Neck', 'Nest', 'Newton', 'New', 'Notch'],
            'O': ['Oaks', 'Orchard', 'Overlook', 'Outpost', 'Outlet', 'Oak', 'Oasis', 'Old'],
            'P': ['Park', 'Pines', 'Plains', 'Point', 'Pond', 'Port', 'Plaza', 'Pass'],
            'Q': ['Quarry', 'Quarter', 'Quay', 'Queen', 'Quarters', 'Quayside', 'Quest', 'Quince'],
            'R': ['Ridge', 'River', 'Rock', 'Run', 'Ranch', 'Rapids', 'Reach', 'Rest'],
            'S': ['Springs', 'Shore', 'Shores', 'South', 'Station', 'Summit', 'Shire', 'Side'],
            'T': ['Town', 'Terrace', 'Trace', 'Trail', 'Township', 'Tower', 'Thicket', 'Timber'],
            'U': ['Union', 'Uplands', 'Upper', 'Underwood', 'Unity', 'University', 'Upton', 'Utopia'],
            'V': ['Vale', 'Valley', 'View', 'Villa', 'Village', 'Vista', 'Ville', 'Vineyards'],
            'W': ['West', 'Water', 'Waters', 'Way', 'Wells', 'Wood', 'Woods', 'Wick'],
            'X': ['Xanadu', 'X-Roads', 'Xing', 'Xavier', 'Xeric', 'Xenia', 'Xenophon', 'Xyst'],
            'Y': ['Yard', 'Yonder', 'York', 'Yards', 'Yew', 'Yews', 'Yale', 'Yarmouth'],
            'Z': ['Zone', 'Zenith', 'Zephyr', 'Zion', 'Zinc', 'Zodiac', 'Zona', 'Zuni'],
        }

        # Naming patterns for settlements
        PATTERNS = [
            "{name}ville",           # Karlville
            "{name}ton",             # Karlton
            "{name}burg",            # Karlburg
            "{name}wood",            # Karlwood
            "{name} {place}",        # Karl Creek
            "{name}'s {place}",      # Karl's Crossing
            "{place} of {name}",     # City of Karl
            "New {name}",            # New Karl
            "Old {name}",            # Old Karl
            "Little {name}",         # Little Karl
            "Upper {name}",          # Upper Karl
            "Lower {name}",          # Lower Karl
            "East {name}",           # East Karl
            "West {name}",           # West Karl
            "North {name}",          # North Karl
            "South {name}",          # South Karl
        ]

        # Shorter patterns for descriptors (avoid double suffixes)
        DESCRIPTOR_PATTERNS = [
            "{name} {place}",        # Karl Creek
            "{name}'s {place}",      # Karl's Crossing
            "{place} of {name}",     # City of Karl
        ]

        if not name:
            return ""
        
        first_letter = name[0].upper()
        
        # Get possible descriptors for this letter
        descriptors = SETTLEMENT_DESCRIPTORS.get(first_letter, ['Place', 'Point', 'Plaza'])
        
        # Decide whether to use suffix or descriptor pattern
        if use_suffix is None:
            use_suffix = random.choice([True, False])
        
        if use_suffix:
            # Use simple suffix patterns (first 7 patterns)
            patterns = PATTERNS[:7]
            if pattern is not None and 0 <= pattern < len(patterns):
                template = patterns[pattern]
            else:
                template = random.choice(patterns)
            
            if "{place}" in template:
                place = descriptor if descriptor and descriptor in descriptors else random.choice(descriptors)
                return template.format(name=name, place=place)
            else:
                return template.format(name=name)
        else:
            # Use directional/size prefix patterns (last 9 patterns)
            patterns = PATTERNS[7:]
            if pattern is not None and 0 <= pattern < len(patterns):
                template = patterns[pattern]
            else:
                template = random.choice(patterns)
            return template.format(name=name)

    @staticmethod
    def _render_attrs(attrs: dict = None) -> str:
        """Render a dict of HTML/HTMX attributes into an SVG attribute string."""
        if not attrs:
            return ""
        return " " + " ".join(f"{k}='{v}'" for k, v in attrs.items())

    @staticmethod
    def decode(s: str) -> 'CountryFlag':
        """Decode CountryFlag from string."""
        parts = s.split('|')
        primary = parts[0]
        name = parts[1]
        year = int(parts[2])
        if len(parts) > 3:
            patternIndex = int(parts[3])
        else:
            patternIndex = 0
        
        # Convert hex back to RGB tuple (0-1 range)
        rgb = plt.matplotlib.colors.to_rgb(primary)
        
        return CountryFlag(rgb, name=name, year=year,patternIndex=patternIndex)

In [ ]:
#| export
@patch
def encode(self: CountryFlag) -> str:
    """Encode CountryFlag to a single line string."""
    # Format: primary|name|year
    return f"{self.primary}|{self.name}|{self.year}|{self.patternIndex}"

In [ ]:
#| export
@patch
def plain(self:CountryFlag,name,width=3):
    return StyleCSS(name,fill=self.primary,stroke=self.comp,stroke_width=width)

@patch
def kingStyle(self:CountryFlag,name,width=2):
    #saturation = 0.7
    style = self.plain(name,width)
    style.name = f"country_{name}"
    #style.properties["fill"] = style.desaturate(saturation).properties["fill"]
    style.properties["opacity"] = 0.7
    return style

@patch
def contrastStyle(self:CountryFlag,name,width=1.5):
    return StyleCSS(
            f"contrast_{name}",
            fill=self.comp,
            stroke="#000",
            stroke_width=width
        )

@patch
def labelStyle(self:CountryFlag, name, width=0):
    return StyleCSS(
        f"contrast_{name}",
        fill=self.darkPrimary,
        stroke="none",
        stroke_width=width
    )


I want to thing in terms of color blindness for our colors. I think primary should be more of our background color and should be slightly lighter and comp could be for text and slightly darker and could be used for labels

Is this at that ratio?

In [ ]:
flags = CountryFlag.seaborn("husl",4)
for flag in flags:
    print(flag.name , flag.capital)


In [ ]:
hexCount = 9
radius = 30
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    style = flat.plain(f"flag_{i}",width=10)
    sampleHex = Hex(radius=30, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
#| export


@patch
def circlePattern(self:CountryFlag, id):
        """Generate a circle pattern definition"""
        content = f'<rect width=120 height= 90 fill="{self.comp}"/><circle cx="{50}" cy="{45}" r="{30}" fill="{self.primary}"/>'
        return SVGDef("pattern", id, content, 
                    width=120, height=90, 
                    patternUnits="userSpaceOnUse")


In [ ]:
#| export
@patch
def triPattern(self:CountryFlag, id):
        """Generate a circle pattern definition"""
        content = f'''<rect width=96 height= 96 fill="{self.primary}"/>
        <circle cx="{48}" cy="{48}" r="{48}" fill="{self.comp}"/>
        <circle cx="{48}" cy="{48}" r="{32}" fill="{self.tri1}"/>
        <circle cx="{48}" cy="{48}" r="{16}" fill="{self.tri2}"/>'''
        
        return SVGDef("pattern", id, content, 
                    width=96, height=96, 
                    patternUnits="userSpaceOnUse")

In [ ]:
#| export
@patch
def swirl(self:CountryFlag,id):

    content = f"""<g  fill='{self.primary}'><rect width=400 height=400 fill='{self.primary}' /></g>
<g  fill='{self.comp}' fill-opacity='1'><path d='M400 58.58c-38.95 0-74.21 15.74-99.79 41.21c-25.61 25.72-61.05 41.64-100.21 41.64s-74.6-15.92-100.21-41.64C74.21 74.32 38.95 58.58 0 58.58c-78.11 0-141.42 63.32-141.42 141.42S-78.11 341.42 0 341.42c38.95 0 74.21-15.74 99.79-41.21c25.61-25.72 61.05-41.64 100.21-41.64s74.6 15.92 100.21 41.64c25.58 25.47 60.84 41.21 99.79 41.21c78.11 0 141.42-63.32 141.42-141.42S478.11 58.58 400 58.58z'/><circle  cx='200' cy='1' r='60'/><circle  cx='200' cy='400' r='60'/></g><g  fill='{self.primary}'><circle  cx='0' cy='200' r='60'/><circle  cx='400' cy='200' r='60'/></g>
"""
    pat = SVGDef("pattern", id, content,
                  width=400, height=400,
                  patternUnits="userSpaceOnUse")
    scale = 0.2
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.swirl(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
#| export
@patch
def yin(self:CountryFlag,id):
    content = f"""<rect fill='{self.primary}' width='800' height='800'/>
     	<g id="YinYang" fill="{self.comp}" stroke="none" stroke-width="0" fill-rule="evenodd">
		<title>Yin-Yang, by Adam Stanislav</title>
		<desc>The entire graphic is drawn as a single path filled with black (or any other color you change the value of “fill” in line 4). The other half, usually shown in white is created here as a hole in the path. That means it is completely transparent, and has whatever color its background has. To achieve this, not just with SVG but with other vector formats, any black portion of the path is drawn counterclockwise, any “white” portion clockwise. Also, this graphic is taking advantage of the kappa constant described in my e-book Bézier Circles and other shapes, freely downloadable from https://www.smashwords.com/books/view/483578 .</desc>

		<!-- Note to self: Relative Bézier differences (“c”) are differences of a point from the STARTING point of the curve segment, not from the most recent point. -->
		<path d="M400 0C179.086 0 0 179.086 0 400 0 620.914 179.086 800 400 800 620.914 800 800 620.914 800 400 800 179.086 620.914 0 400 0zM400 10C184.609 10 10 184.609 10 400 10 615.391 184.609 790 400 790 292.304 790 205 682.304 205 600 205 492.3 292.304 400 400 400 507.7 400 600 292.304 600 200 600 92.304 507.7 10 400 10zM400 665c35.895 0 65-29.105 65-65 0-35.895-29.105-65-65-65-35.895 0-65 29.105-65 65 0 35.895 29.105 65 65 65zM400 132c-37.555 0-68 30.445-68 68 0 37.555 30.445 68 68 68 37.555 0 68-30.445 68-68 0-37.555-30.445-68-68-68z" stroke='black' stroke-width='25' stroke-opacity='1' stroke-linecap='square' />
	</g>
  """


    pat = SVGDef("pattern", id, content,
                  width=800, height=800,
                  patternUnits="userSpaceOnUse")
    scale = 0.1 
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.yin(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

How do I fill black in yin

In [ ]:
#| export
@patch
def weave(self:CountryFlag, id):
    
    content = f"""<rect fill='{self.primary}' width='600' height='600'/><path  fill='none' stroke='{self.comp}' stroke-width='55' stroke-opacity='1' stroke-linecap='square' d='M250.5 50.5h-200M250.5 150.5h-200M250.5 250.5h-200M550.5 350.5h-200M550.5 450.5h-200M550.5 550.5h-200M250.5 550.5v-200M150.5 550.5v-200M50.5 550.5v-200M550.5 250.5v-200M450.5 250.5v-200M350.5 250.5v-200'/>
    """
    pat = SVGDef("pattern", id, content,
                  width=600, height=600,
                  patternUnits="userSpaceOnUse")
    scale = 0.25
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
hexCount = 19
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("magma", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.weave(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.yin(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.triPattern(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

In [ ]:
for pat in canvas.definitions:
    print(pat.attributes["id"])

In [ ]:


hexStyles = CountryFlag.seaborn("Accent", hexCount)
hexIndex = 3
name = f"CountrySwirl_{hexIndex}"
patternName = f"{name}_pat"
hexPat = hexStyles[hexIndex].swirl(patternName)

#some drawing setup 
hexStyle = StyleCSS(name, fill=f"url(#{patternName})")
canvas = SVGBuilder()
canvas.width=200 ;canvas.height=200
canvas.add_style(hexStyle)
canvas.add_definition(hexPat)

# add our hex to the canvas
sampleHex = Hex(radius=50,center=MapCord(100,100),style=hexStyle)
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()
canvas.adjust("main",sampleHex.svg())

#show our work
canvas.show()



In [ ]:
#| export
@patch
def sheridan(self:CountryFlag,id):
    content = f"""<rect fill='{self.primary} width='1600' height='900'/><path  fill='{self.tri1}' d='M799 0v0.5c0 0-49.4 40.5-49.4 90s99.8 130.3 99.8 180c0 49.7-99.8 130.5 -99.8 180s99.8 130.3 99.8 180s-99.8 130.5-99.8 180c0 46.5 40.4 89.5 40.4 89.5h9h10h792v-900h-802z'/><path  fill='{self.tri2}' d='M751.6 450.5c0-49.5 99.8-130.3 99.8-180c0-49.7-99.8-130.5-99.8-180s49.4-90 49.4-90v-0.5h-802v900h793c0 0-40.4-43-40.4-89.5c0-49.5 99.8-130.3 99.8-180s-99.8-130.5-99.8-180z'/>"""
    pat = SVGDef("pattern", id, content,
                  width=1600, height=900,
                  patternUnits="userSpaceOnUse")
    scale = 0.1
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat

Lets redo sheridan so it is vertical waves of comp and primary

It does look good as a repeated pattern. Can you build a better version?

Can you make chess pieces svg for our country?

use the built-in chess.svg piece shapes (recolored with CountryFlag colors),
all 6.
we probaly need some sort of scale and center coordinates to place on a map

It might be easier to just take the raw svg and build our own defs and not use the libary (other than for building our functions

piece_svgs = {}
for sym in ["K", "Q", "B", "N", "R", "P"]:
    raw = chess.svg.piece(chess.Piece.from_symbol(sym))
    # Extract just the inner <g> content
    inner = raw.split('>', 1)[1].rsplit('</svg>', 1)[0]
    piece_svgs[sym] = inner
    print(f"--- {sym} ---")
    print(inner, "\n")


## Piece of mind

In [ ]:
#| export
@patch
def flag_hex(flag:CountryFlag, radius=25):
    """Hex with flag pattern fill, built via SVGBuilder."""
    pad = 4
    size = radius * 2 + pad * 2
    center = MapCord(size / 2, size / 2)

    builder = SVGBuilder()
    builder.width = size
    builder.height = size

    # Register flag pattern as a fill
    pat_name = f"fh_{flag.name}"
    builder.add_definition(flag.flagPattern(pat_name, scale=0.08))
    pat_style = StyleCSS(pat_name, fill=f"url(#{pat_name})",
                         stroke=flag.comp, stroke_width=3)
    builder.add_style(pat_style)

    h = Hex(radius=radius, center=center, style=pat_style)
    builder.adjust("hex", h.svg())

    return Div(NotStr(builder.xml()))

In [ ]:
!cat ../../docs/llms*

Can you build something like the flag_hex but with a more traditional flag shape. Use our primitives as you can. we might want to be able to make this fasthtml/htmlx compatiable as well.

In [ ]:
#| export
@patch
def flag_banner(self: CountryFlag, width=150, height=100,
                pattern_fn=None, wavy=False, pole=True, attrs=None):
    """Traditional rectangular flag with optional pole and wavy edge.
    
    Returns SVGBuilder. Use .show() for Jupyter or wrap for FastHTML:
        Div(NotStr(flag.flag_banner().xml()), **htmx_attrs)
    
    Args:
        width/height: flag dimensions
        pattern_fn: 'swirl','weave','triPattern','circlePattern','yin','sheridan'
                    (defaults via patternIndex)
        wavy: wavy right edge via MapPath.make_windy
        pole: draw flagpole on left
        attrs: extra SVG attributes dict for the flag polygon
    """
    PATTERNS = ['swirl', 'weave', 'triPattern', 'circlePattern', 'yin', 'sheridan']
    if pattern_fn is None:
        pattern_fn = PATTERNS[self.patternIndex % len(PATTERNS)]
    
    pad, pole_w, finial_r, pole_extra = 6, 5, 6, 30
    
    # Layout offsets
    fx = pad + (pole_w if pole else 0)
    fy = pad + (finial_r * 2 if pole else 0)
    
    builder = SVGBuilder()
    builder.width = fx + width + pad
    builder.height = fy + height + (pole_extra if pole else 0) + pad
    
    # --- Pattern fill ---
    pat_id = f"fp_{self.name}"
    builder.add_definition(getattr(self, pattern_fn)(pat_id))
    flag_style = StyleCSS(f"flag_{self.name}",
                          fill=f"url(#{pat_id})", stroke=self.comp, stroke_width=2.5)
    builder.add_style(flag_style)
    
    # --- Flag shape via MapPath ---
    tl, tr = MapCord(fx, fy), MapCord(fx + width, fy)
    br, bl = MapCord(fx + width, fy + height), MapCord(fx, fy + height)
    
    if wavy:
        right_edge = MapPath([tr, br]).make_windy(iterations=3, offset_factor=0.12)
        points = [tl, tr] + right_edge.points[1:] + [bl]
    else:
        points = [tl, tr, br, bl]
    
    flag_path = MapPath(points, flag_style)
    builder.adjust("flag", flag_path.drawClosed(CountryFlag._render_attrs(attrs)))
    
    # --- Pole + finial ---
    if pole:
        pole_style = StyleCSS(f"pole_{self.name}",
                              stroke=self.darkPrimary, stroke_width=pole_w,
                              stroke_linecap="round", fill=self.darkPrimary)
        builder.add_style(pole_style)
        px = pad + pole_w / 2
        builder.adjust("pole",
            f'<line x1="{px}" y1="{pad + finial_r}" '
            f'x2="{px}" y2="{fy + height + pole_extra}" '
            f'class="{pole_style.name}"/>'
            f'<circle cx="{px}" cy="{pad + finial_r}" '
            f'r="{finial_r}" class="{pole_style.name}"/>')
    
    return builder


In [ ]:
#| export
@patch
def flag_html(self: CountryFlag, width=150, height=100,
              pattern_fn=None, wavy=False, pole=True, **htmx_attrs):
    """FastHTML-compatible flag. Pass any HTMX attrs as kwargs.
    
    Example: flag.flag_html(hx_get='/click', hx_target='#info')
    """
    builder = self.flag_banner(width=width, height=height,
                               pattern_fn=pattern_fn, wavy=wavy, pole=pole)
    return Div(NotStr(builder.xml()), **htmx_attrs)


In [ ]:
# Demo: show all pattern types
flags = CountryFlag.seaborn("husl", 6)
patterns = ['swirl', 'weave', 'triPattern', 'circlePattern', 'yin', 'sheridan']

canvas = SVGBuilder()
cols = len(patterns)
fw, fh = 150, 100
canvas.width = cols * (fw + 20) + 20
canvas.height = fh + 80

for i, (flag, pat) in enumerate(zip(flags, patterns)):
    sub = flag.flag_banner(width=fw, height=fh, pattern_fn=pat, pole=True, wavy=(i % 2 == 1))
    # Offset each flag horizontally
    x_off = 10 + i * (fw + 20)
    canvas.definitions.extend(sub.definitions)
    canvas.styles.update(sub.styles)
    wrapped = f'<g transform="translate({x_off}, 0)">'
    for layer in sub.layers:
        wrapped += layer.body
    wrapped += '</g>'
    canvas.adjust(f"flag_{i}", wrapped)

canvas.show()


In [ ]:
#| export
@patch
def kingPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
              piece_id: str = None, fill: str = None, attrs: dict = None):
    """Place a chess king at center. attrs: dict of HTML/HTMX attributes for interactivity.
    For instance:
    Then usage looks like:
    
flag.kingPiece(center, scale=1.5, attrs={
    'hx-get': '/piece/king/clicked',
    'hx-target': '#game-panel',
    'hx-swap': 'innerHTML',
    'data-piece': 'king',
    'data-player': flag.name
})


    
    """
    if piece_id is None:
        piece_id = f"king_{self.name}"
    if fill is None:
        fill = self.primary
    
    paths = f'''<path d="M22.5 11.63V6M20 8h5" stroke-linejoin="miter" />
<path d="M22.5 25s4.5-7.5 3-10.5c0 0-1-2.5-3-2.5s-3 2.5-3 2.5c-1.5 3 3 10.5 3 10.5"
      fill="{fill}" stroke-linecap="butt" stroke-linejoin="miter" />
<path d="M11.5 37c5.5 3.5 15.5 3.5 21 0v-7s9-4.5 6-10.5c-4-6.5-13.5-3.5-16 4V27v-3.5c-3.5-7.5-13-10.5-16-4-3 6 5 10 5 10V37z"
      fill="{fill}" />
<path d="M11.5 30c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    
    return f'''<g id="{piece_id}" fill="none" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''


In [ ]:
#| export
@patch
def queenPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
               piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"queen_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<path d="M8 12a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM24.5 7.5a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM41 12a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM16 8.5a2 2 0 1 1-4 0 2 2 0 1 1 4 0zM33 9a2 2 0 1 1-4 0 2 2 0 1 1 4 0z" />
<path d="M9 26c8.5-1.5 21-1.5 27 0l2-12-7 11V11l-5.5 13.5-3-15-3 15-5.5-14V25L7 14l2 12zM9 26c0 2 1.5 2 2.5 4 1 1.5 1 1 .5 3.5-1.5 1-1.5 2.5-1.5 2.5-1.5 1.5.5 2.5.5 2.5 6.5 1 16.5 1 23 0 0 0 1.5-1 0-2.5 0 0 .5-1.5-1-2.5-.5-2.5-.5-2 .5-3.5 1-2 2.5-2 2.5-4-8.5-1.5-18.5-1.5-27 0z" stroke-linecap="butt" />
<path d="M11.5 30c3.5-1 18.5-1 22 0M12 33.5c6-1 15-1 21 0" fill="none" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}" fill="{fill}" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''

@patch
def bishopPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
                piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"bishop_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<g fill="{fill}" stroke-linecap="butt">
<path d="M9 36c3.39-.97 10.11.43 13.5-2 3.39 2.43 10.11 1.03 13.5 2 0 0 1.65.54 3 2-.68.97-1.65.99-3 .5-3.39-.97-10.11.46-13.5-1-3.39 1.46-10.11.03-13.5 1-1.354.49-2.323.47-3-.5 1.354-1.94 3-2 3-2zM15 32c2.5 2.5 12.5 2.5 15 0 .5-1.5 0-2 0-2 0-2.5-2.5-4-2.5-4 5.5-1.5 6-11.5-5-15.5-11 4-10.5 14-5 15.5 0 0-2.5 1.5-2.5 4 0 0-.5.5 0 2zM25 8a2.5 2.5 0 1 1-5 0 2.5 2.5 0 1 1 5 0z" />
</g>
<path d="M17.5 26h10M15 30h15m-7.5-14.5v5M20 18h5" stroke-linejoin="miter" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}" fill="none" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''

@patch
def knightPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
                piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"knight_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<path d="M 22,10 C 32.5,11 38.5,18 38,39 L 15,39 C 15,30 25,32.5 23,18"
      fill="{fill}" stroke="{self.comp}" />
<path d="M 24,18 C 24.38,20.91 18.45,25.37 16,27 C 13,29 13.18,31.34 11,31 C 9.958,30.06 12.41,27.96 11,28 C 10,28 11.19,29.23 10,30 C 9,30 5.997,31 6,26 C 6,24 12,14 12,14 C 12,14 13.89,12.1 14,10.5 C 13.27,9.506 13.5,8.5 13.5,7.5 C 14.5,6.5 16.5,10 16.5,10 L 18.5,10 C 18.5,10 19.28,8.008 21,7 C 22,7 22,10 22,10"
      fill="{fill}" stroke="{self.comp}" />
<path d="M 9.5 25.5 A 0.5 0.5 0 1 1 8.5,25.5 A 0.5 0.5 0 1 1 9.5 25.5 z"
      fill="{self.comp}" stroke="{self.comp}" />
<path d="M 15 15.5 A 0.5 1.5 0 1 1 14,15.5 A 0.5 1.5 0 1 1 15 15.5 z"
      transform="matrix(0.866,0.5,-0.5,0.866,9.693,-5.173)"
      fill="{self.comp}" stroke="{self.comp}" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}" fill="none" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''

@patch
def rookPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
              piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"rook_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<path d="M9 39h27v-3H9v3zM12 36v-4h21v4H12zM11 14V9h4v2h5V9h5v2h5V9h4v5" stroke-linecap="butt" />
<path d="M34 14l-3 3H14l-3-3" />
<path d="M31 17v12.5H14V17" stroke-linecap="butt" stroke-linejoin="miter" />
<path d="M31 29.5l1.5 2.5h-20l1.5-2.5" />
<path d="M11 14h23" fill="none" stroke-linejoin="miter" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}" fill="{fill}" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''

@patch
def pawnPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, 
              piece_id: str = None, fill: str = None, attrs: dict = None):
    if piece_id is None: piece_id = f"pawn_{self.name}"
    if fill is None: fill = self.primary
    
    paths = f'''<path d="M22.5 9c-2.21 0-4 1.79-4 4 0 .89.29 1.71.78 2.38C17.33 16.5 16 18.59 16 21c0 2.03.94 3.84 2.41 5.03-3 1.06-7.41 5.55-7.41 13.47h23c0-7.92-4.41-12.41-7.41-13.47 1.47-1.19 2.41-3 2.41-5.03 0-2.41-1.33-4.5-3.28-5.62.49-.67.78-1.49.78-2.38 0-2.21-1.79-4-4-4z"
      fill="{fill}" stroke="{self.comp}" stroke-width="1.5" stroke-linecap="round" />'''

    tx, ty = center.x, center.y
    extra = CountryFlag._render_attrs(attrs)
    return f'''<g id="{piece_id}"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)"
   style="cursor:pointer"{extra}>
{paths}
</g>'''


I want to add fasthtml style htmx clicking to these

Can you write the other methods? Did I patch King piece correctly

In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
canvas = SVGBuilder()
canvas.width, canvas.height = 400, 80

pieces = [flag.kingPiece, flag.queenPiece, flag.bishopPiece, 
          flag.knightPiece, flag.rookPiece, flag.pawnPiece]

for i, fn in enumerate(pieces):
    canvas.adjust(f"p{i}", fn(MapCord(40 + i*60, 40), scale=1.2))

canvas.show()


In [ ]:
hexCount = 9
radius = 100
padding = 10
itemWidth = (radius * 2 + padding)

hexStyles = CountryFlag.seaborn("Accent", hexCount)
canvas = SVGBuilder()
canvas.width = hexCount * itemWidth
canvas.height = itemWidth + padding

for i , flat in enumerate(hexStyles):
    name = f"CountryStar_{i}"

    patternName = f"{name}_pat"
    pattern = flat.sheridan(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    sampleHex = Hex(radius=radius, center=MapCord((padding+radius) + i * itemWidth, (padding + radius)), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pattern)

# Apply animation
layer_names = [f"hex-{i}" for i in range(hexCount)]
#anim = LoopingLayerAnimation(layer_names, visible_count=3, step_duration=0.5, fade_duration=0.1, dim_opacity=0)
#apply_looping_animation(canvas, anim)
canvas.show()

So we have a bunch of flag patterns (swirl, yin, sheridan), but ultimately we need a dispatch function using the patternIndex for a flag. Can you build? any other interesing kind patterns to use

In [ ]:
#| export
@patch
def chevron(self:CountryFlag, id):
    """Zigzag chevron pattern"""
    content = f"""<rect fill='{self.primary}' width='200' height='200'/>
    <path fill='{self.comp}' d='M0 0 L100 50 L200 0 L200 50 L100 100 L0 50Z'/>
    <path fill='{self.comp}' d='M0 100 L100 150 L200 100 L200 150 L100 200 L0 150Z'/>"""
    pat = SVGDef("pattern", id, content, width=200, height=200, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.25)'
    return pat

@patch
def scales(self:CountryFlag, id):
    """Fish-scale / overlapping semicircle pattern"""
    content = f"""<rect fill='{self.primary}' width='100' height='100'/>
    <circle cx='50' cy='100' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>
    <circle cx='0' cy='50' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>
    <circle cx='100' cy='50' r='50' fill='none' stroke='{self.comp}' stroke-width='12'/>"""
    pat = SVGDef("pattern", id, content, width=100, height=100, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.35)'
    return pat

@patch
def diamonds(self:CountryFlag, id):
    """Argyle diamond grid"""
    content = f"""<rect fill='{self.primary}' width='200' height='200'/>
    <polygon points='100,0 200,100 100,200 0,100' fill='{self.comp}'/>
    <line x1='0' y1='0' x2='200' y2='200' stroke='{self.tri1}' stroke-width='4'/>
    <line x1='200' y1='0' x2='0' y2='200' stroke='{self.tri1}' stroke-width='4'/>"""
    pat = SVGDef("pattern", id, content, width=200, height=200, patternUnits="userSpaceOnUse")
    pat.attributes['patternTransform'] = 'scale(0.2)'
    return pat


In [ ]:
#| export
@patch
def fanBlade(self:CountryFlag, id):
    blade_id = f"{id}_b"
    content = f"""<rect width='100' height='100' fill='{self.primary}'/>
    <g stroke='none'>
        <path id='{blade_id}' fill='{self.comp}' d='M75 50c-6.9 0-12.5-5.6-12.5-12.5S68.1 25 75 25c0-6.9-5.6-12.5-12.5-12.5S50 18.1 50 25s-5.6 12.5-12.5 12.5S25 31.9 25 25c-6.9 0-12.5 5.6-12.5 12.5S18.1 50 25 50s12.5 5.6 12.5 12.5S31.9 75 25 75c0 6.9 5.6 12.5 12.5 12.5S50 81.9 50 75s5.6-12.5 12.5-12.5S75 68.1 75 75c6.9 0 12.5-5.6 12.5-12.5S81.9 50 75 50z'/>
        <use href='#{blade_id}' x='-50' y='50'/>
        <use href='#{blade_id}' x='-50' y='-50'/>
        <use href='#{blade_id}' x='50' y='-50'/>
        <use href='#{blade_id}' x='50' y='50'/>
    </g>"""

    pat = SVGDef("pattern", id, content,
                  width=100, height=100,
                  patternUnits="userSpaceOnUse")
    scale = 0.4
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def sheridan(self:CountryFlag, id):
    W, H = 200, 200
    A = 30  # wave amplitude
    cx1, cx2 = 50, 150  # wave boundary centers

    content = f"""<rect width='{W}' height='{H}' fill='{self.primary}'/>
    <path fill='{self.comp}' d='
        M {cx1},0
        Q {cx1+A},{H//4} {cx1},{H//2}
        Q {cx1-A},{3*H//4} {cx1},{H}
        L {cx2},{H}
        Q {cx2-A},{3*H//4} {cx2},{H//2}
        Q {cx2+A},{H//4} {cx2},0
        Z'/>"""

    pat = SVGDef("pattern", id, content,
                  width=W, height=H,
                  patternUnits="userSpaceOnUse")
    scale = 0.5
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def leafy(self:CountryFlag, id):
    content = f"""<rect fill='{self.primary}' width='250' height='200'/>
    <g fill='{self.comp}'>
        <path d='M161 100c0 33.22-36 34.14-36 60c0-25.86-36-26.78-36-60s36-34.14 36-60C125 65.86 161 66.78 161 100z'/>
        <path d='M35 0C35 33.22 0 34.14 0 60c0-25.86-65-90-65-90S35-33.22 35 0z'/>
        <path d='M35 200c0 33.22-100 30-100 30s65-64.14 65-90C0 165.86 35 166.78 35 200z'/>
        <path d='M315-30c0 0-65 64.14-65 90c0-25.86-35-26.78-35-60S315-30 315-30z'/>
        <path d='M315 230c0 0-100 3.22-100-30s35-34.14 35-60C250 165.86 315 230 315 230z'/>
    </g>
    <g fill='none' stroke='{self.tri1}' stroke-width='40'>
        <path d='M39.53-98.77c0 40.43 46.61 56.95 46.61 98.85s-47.19 58.59-47.19 99.3s47.19 57.24 47.19 99.3s-46.61 58.41-46.61 98.85'/>
        <path d='M211.72 297.53c0-40.71-47.19-57.24-47.19-99.3s46.61-58.41 46.61-98.85c0-40.43-46.61-56.95-46.61-98.85s47.19-58.59 47.19-99.3'/>
    </g>"""

    pat = SVGDef("pattern", id, content,
                  width=250, height=200,
                  patternUnits="userSpaceOnUse")
    scale = 0.35
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
#| export
@patch
def angled(self:CountryFlag, id):
    content = f"""<rect fill='{self.primary}' width='2000' height='1000'/>
<path fill='{self.comp}' d='M1500 100l-100 100v200l200 200V200zm500 100V0l-200 200v200zM2000 1000l-144-144 144-56zm-200-600l-400 400 200 200V800l400-400zM1200 800v200h-200zm100-300l-300 300h200l200-200zm-200-400L1200 0h-200v200l200 200h200z'/>
<path fill='{self.tri1}' d='M1600 0h-200l200 200v400l200-200V200h-200L1800 0zM2000 600V400l-200 200v200zM800 0H600l200 200 400 400 100-100zm400 0l-100 100 300 300V200zm200 800h-200l200 200h200zM1800 800l-200 200h200l200-200zm-800 0L600 400H400l400 400-200 200h200l200-200zM600 800L400 600V400L200 600 0 800v200h400l100-100zm0-800H200l200 200z'/>
<path fill='{self.comp}' d='M800 200V0L400 400h200l100-100 300 300h200zM0 200V0h200zm600 400L400 400v200l200 200h200zM400 200L300 100 0 400l200 200V400zm100 700L200 600 100 700l300 300zm-300 100H0V800z'/>"""

    pat = SVGDef("pattern", id, content,
                  width=2000, height=1000,
                  patternUnits="userSpaceOnUse")
    scale = 0.08
    pat.attributes['patternTransform'] = f'scale({scale})'
    return pat


In [ ]:
def _settlement_pattern(pat_id: str, flag: 'CountryFlag', size: int = 40) -> str:
    """Generate an SVG <pattern> element from a CountryFlag's colors + patternIndex."""
    s = size
    h = s // 2
    p, c, t1, t2 = flag.primary, flag.comp, flag.tri1, flag.tri2
    idx = flag.patternIndex % 6

    if idx == 0:    # horizontal bicolor
        inner = (f'<rect width="{s}" height="{h}" fill="{p}"/>'
                 f'<rect y="{h}" width="{s}" height="{h}" fill="{c}"/>')
    elif idx == 1:  # vertical bicolor
        inner = (f'<rect width="{h}" height="{s}" fill="{p}"/>'
                 f'<rect x="{h}" width="{h}" height="{s}" fill="{c}"/>')
    elif idx == 2:  # tricolor horizontal
        t = s // 3
        inner = (f'<rect width="{s}" height="{t}" fill="{p}"/>'
                 f'<rect y="{t}" width="{s}" height="{t}" fill="{t1}"/>'
                 f'<rect y="{2*t}" width="{s}" height="{t}" fill="{c}"/>')
    elif idx == 3:  # quarters
        inner = (f'<rect width="{h}" height="{h}" fill="{p}"/>'
                 f'<rect x="{h}" width="{h}" height="{h}" fill="{c}"/>'
                 f'<rect y="{h}" width="{h}" height="{h}" fill="{t1}"/>'
                 f'<rect x="{h}" y="{h}" width="{h}" height="{h}" fill="{t2}"/>')
    elif idx == 4:  # cross on solid
        w = s // 5
        inner = (f'<rect width="{s}" height="{s}" fill="{p}"/>'
                 f'<rect x="{h - w//2}" width="{w}" height="{s}" fill="{c}"/>'
                 f'<rect y="{h - w//2}" width="{s}" height="{w}" fill="{c}"/>')
    else:           # diagonal split
        inner = (f'<rect width="{s}" height="{s}" fill="{p}"/>'
                 f'<polygon points="0,0 {s},0 0,{s}" fill="{c}"/>')

    return (f'<pattern id="{pat_id}" width="{s}" height="{s}" '
            f'patternUnits="userSpaceOnUse">{inner}</pattern>')


In [ ]:
#| export
@patch
def bicolorH(self:CountryFlag, id):
    """Horizontal bicolor flag"""
    s = 40; h = s // 2
    content = f'<rect width="{s}" height="{h}" fill="{self.primary}"/><rect y="{h}" width="{s}" height="{h}" fill="{self.comp}"/>'
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def bicolorV(self:CountryFlag, id):
    """Vertical bicolor flag"""
    s = 40; h = s // 2
    content = f'<rect width="{h}" height="{s}" fill="{self.primary}"/><rect x="{h}" width="{h}" height="{s}" fill="{self.comp}"/>'
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def tricolorH(self:CountryFlag, id):
    """Horizontal tricolor flag"""
    s = 42; t = s // 3
    content = (f'<rect width="{s}" height="{t}" fill="{self.primary}"/>'
               f'<rect y="{t}" width="{s}" height="{t}" fill="{self.tri1}"/>'
               f'<rect y="{2*t}" width="{s}" height="{t}" fill="{self.comp}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")

@patch
def quarters(self:CountryFlag, id):
    """Quartered flag"""
    s = 40; h = s // 2
    content = (f'<rect width="{h}" height="{h}" fill="{self.primary}"/>'
               f'<rect x="{h}" width="{h}" height="{h}" fill="{self.comp}"/>'
               f'<rect y="{h}" width="{h}" height="{h}" fill="{self.tri1}"/>'
               f'<rect x="{h}" y="{h}" width="{h}" height="{h}" fill="{self.tri2}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")


In [ ]:
#| export

@patch
def crossFlag(self:CountryFlag, id):
    """Cross on solid background with center circle in tri2"""
    s = 40; h = s // 2; w = s // 5; r = w 
    content = (f'<rect width="{s}" height="{s}" fill="{self.primary}"/>'
               f'<rect x="{h - w//2}" width="{w}" height="{s}" fill="{self.comp}"/>'
               f'<rect y="{h - w//2}" width="{s}" height="{w}" fill="{self.comp}"/>'
               f'<circle cx="{h}" cy="{h}" r="{r}" fill="{self.tri2}"/>')
    return SVGDef("pattern", id, content, width=s, height=s, patternUnits="userSpaceOnUse")


In [ ]:
#| export
@patch
def flagPattern(self:CountryFlag, id, scale=None):
    """Dispatch to pattern based on patternIndex, with optional scale override"""
    patterns = [
        self.circlePattern,  # 0
        self.triPattern,     # 1
        self.swirl,          # 2
        self.yin,            # 3
        self.weave,          # 4
        self.sheridan,       # 5
        self.chevron,        # 6
        self.scales,         # 7
        self.diamonds,       # 8
        self.fanBlade,       # 9
        self.leafy,          # 10
        self.bicolorH,       # 11
        self.bicolorV,       # 12
        self.tricolorH,      # 13
        self.quarters,       # 14
        self.crossFlag,      # 15
    ]
    idx = self.patternIndex % len(patterns)
    pat = patterns[idx](id)
    
    if scale is not None:
        # Parse existing scale from patternTransform (default 1.0 if not set)
        existing_transform = pat.attributes.get('patternTransform', 'scale(1)')
        m = re.search(r'scale\(([\d.]+)\)', existing_transform)
        base_scale = float(m.group(1)) if m else 1.0
        pat.attributes['patternTransform'] = f'scale({base_scale * scale})'
    
    return pat


Can we update flagPattern to take a scale parameters. sometimes we are going to need to shrink the flag

So the tricky part is that each of the underlying flags have their own scale. it needs to read the existing scale and then modify it

## Packets

Can you make the gallery have 3 rows

## Gallery

In [ ]:
hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60  # hex height + label space

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    sampleHex = Hex(radius=radius, center=MapCord(cx, cy), style=style)
    canvas.adjust(f"hex-{i}", sampleHex.svg())
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="sans-serif" fill="#333">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


Can you update the gallery so it uses king pieces instead of hexes

@patch
def kingPiece(self: CountryFlag, center: MapCord, scale: float = 1.0, piece_id: str = None, fill: str = None):
    """Place a chess king at center, colored with CountryFlag colors.
    The raw piece is drawn in a 45x45 box centered ~(22.5, 22.5).
    fill: override body fill (e.g. 'url(#pattern_id)' for pattern fill)"""
    
    if piece_id is None:
        piece_id = f"king_{self.name}"
    if fill is None:
        fill = self.primary
    
    paths = f'''<path d="M22.5 11.63V6M20 8h5" stroke-linejoin="miter" />
<path d="M22.5 25s4.5-7.5 3-10.5c0 0-1-2.5-3-2.5s-3 2.5-3 2.5c-1.5 3 3 10.5 3 10.5"
      fill="{fill}" stroke-linecap="butt" stroke-linejoin="miter" />
<path d="M11.5 37c5.5 3.5 15.5 3.5 21 0v-7s9-4.5 6-10.5c-4-6.5-13.5-3.5-16 4V27v-3.5c-3.5-7.5-13-10.5-16-4-3 6 5 10 5 10V37z"
      fill="{fill}" />
<path d="M11.5 30c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0m-21 3.5c5.5-3 15.5-3 21 0" />'''

    tx, ty = center.x, center.y
    
    return f'''<g id="{piece_id}" fill="none" fill-rule="evenodd"
   stroke="{self.comp}" stroke-width="1.5"
   stroke-linecap="round" stroke-linejoin="round"
   transform="translate({tx},{ty}) scale({scale}) translate(-22.5,-22.5)">
{paths}
</g>'''


In [ ]:
hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

# Background rects so the flag pattern shows behind the king
for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName,scale=0.1)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    # Draw a colored circle as backdrop, then the king on top
    king_scale = 3.0
    backdrop = f'<circle cx="{cx}" cy="{cy}" r="{radius * 0.7}" fill="white" stroke="black" stoke-width="1"/>'
    pat_fill = f"url(#{patternName})"
    king_svg = flag.knightPiece(MapCord(cx, cy), scale=king_scale, piece_id=f"king_{i}", fill=pat_fill)

    
    canvas.adjust(f"backdrop-{i}", backdrop)
    canvas.adjust(f"king-{i}", king_svg)
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="sans-serif" fill="#333">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


Can we do the label for the demo using the flag labelStyle?

Can you rewrite it?

In [ ]:
??SVGBuilder.add_font

In [ ]:
hexCount = 16
radius = 80
padding = 20
itemWidth = (radius * 2 + padding)
cols = 6
rowHeight = radius * 2 + 60

canvas = SVGBuilder()
canvas.width = cols * itemWidth
canvas.height = 3 * rowHeight + padding
canvas.add_font('Cinzel')

pattern_names = [
    "circle", "tri", "swirl", "yin", "weave",
    "sheridan", "chevron", "scales", "diamonds", "fanBlade", "leafy",
    "bicolorH", "bicolorV", "tricolorH", "quarters", "crossFlag"
]

flag = CountryFlag.seaborn("husl", 1)[0]

for i in range(hexCount):
    flag.patternIndex = i
    name = f"gallery_{i}"
    patternName = f"{name}_pat"
    
    row = i // cols
    col = i % cols
    
    pat = flag.flagPattern(patternName, scale=0.1)
    style = StyleCSS(name, fill=f"url(#{patternName})")
    cx = (padding + radius) + col * itemWidth
    cy = padding + radius + row * rowHeight
    
    king_scale = 3.0
    backdrop = f'<circle cx="{cx}" cy="{cy}" r="{radius * 0.7}" fill="white" stroke="black" stroke-width="1"/>'
    pat_fill = f"url(#{patternName})"
    king_svg = flag.knightPiece(MapCord(cx, cy), scale=king_scale, piece_id=f"king_{i}", fill=pat_fill)

    canvas.adjust(f"backdrop-{i}", backdrop)
    canvas.adjust(f"king-{i}", king_svg)
    canvas.add_style(style)
    canvas.add_definition(pat)
    
    # Label using flag's labelStyle
    lbl_style = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl_style)
    label = f'<text x="{cx}" y="{cy + radius + 20}" text-anchor="middle" font-size="14" font-family="\'Cinzel\', sans-serif" class="{lbl_style.name}">{pattern_names[i]}</text>'
    canvas.adjust(f"label-{i}", label)

canvas.show()


So the following is some downstream code which I think should be improved since I think we could have a factory function from the flag that is put onto a piece. I do think we are going to have to come up with what works at various sizes. I love the patterns so far for the larger pieces, but we are going to need another for items in a list and another for a piece on a board (say 20x20 or smaller).

```
_PIECE_RENDERERS = {
    PieceType.PAWN:   'pawnPiece',
    PieceType.KNIGHT: 'knightPiece',
    PieceType.ROOK:   'rookPiece',
    PieceType.BISHOP: 'bishopPiece',
    PieceType.QUEEN:  'queenPiece',
    PieceType.KING:   'kingPiece',
}

_PIECE_ICONS = {
    PieceType.PAWN:   "♟", PieceType.KNIGHT: "♞", PieceType.ROOK:   "♜",
    PieceType.BISHOP: "♝", PieceType.QUEEN:  "♛", PieceType.KING:   "♚",
}


@patch
def _ensure_pattern(self: Piece, grid: HexGrid, pat_scale: float = 0.1) -> str:
    """Register flag pattern on grid builder, return CSS fill reference.
    Falls back to flag.primary solid color if no flag."""
    if not self.flag: return "#888"
    pat_name = f"piece_pat_{self.id[:8]}"
    pat = self.flag.flagPattern(pat_name, scale=pat_scale)
    grid.builder.add_definition(pat)
    return f"url(#{pat_name})"


@patch
def draw_svg(self: Piece, grid: HexGrid, hex_idx: int = None,
             scale: float = None, opacity: float = 1.0,
             pat_fill: str = None, suffix: str = "") -> str:
    """Render this piece's chess glyph as SVG at a hex position.

    Args:
        grid:     HexGrid to draw on (needed for hex centres + builder)
        hex_idx:  fine grid index to draw at (defaults to self.location)
        scale:    piece scale (defaults to ~75% of hex diameter)
        opacity:  1.0 for solid, 0.4 for ghost, etc.
        pat_fill: override the fill (otherwise registers flag pattern)
        suffix:   appended to piece_id for uniqueness (e.g. '_ghost')
    """
    idx = hex_idx if hex_idx is not None else self.location
    if idx is None or idx < 0 or idx >= len(grid.hexes): return ""

    center = grid.hexes[idx].center
    renderer = _PIECE_RENDERERS.get(self.piece_type, 'pawnPiece')
    if not self.flag: return ""

    if scale is None:
        scale = (grid.radius * 0.75) / 22.5  # chess SVGs ~22.5px baseline

    fill = pat_fill or self._ensure_pattern(grid)
    pid  = f"piece_{self.id[:8]}{suffix}"

    svg = getattr(self.flag, renderer)(
        MapCord(center.x, center.y),
        scale=scale,
        piece_id=pid,
        fill=fill,
    )

    # Wrap in a group with opacity if not fully opaque
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>\n'

    return svg


@patch
def draw_ghost(self: Piece, grid: HexGrid, hex_idx: int,
               scale: float = None, opacity: float = 0.4) -> str:
    """Draw a faded copy at a projected future position."""
    s = scale or (grid.radius * 0.75) / 22.5
    return self.draw_svg(grid, hex_idx=hex_idx,
                         scale=s * 0.7, opacity=opacity,
                         suffix="_ghost")
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great observation — at small scales, those intricate patterns (swirl, leafy, fanBlade etc.) become visual mush. You really need a **three-tier approach**:

**`'large'`** (~scale 3+, gallery/detail): Full `flagPattern` dispatch — all 16 patterns look great here.

**`'list'`** (~scale 1–2, sidebar/inventory ~40px): Only the simpler patterns work — bicolorH, bicolorV, tricolorH, quarters, crossFlag, chevron. You could remap `patternIndex` to this subset.

**`'board'`** (~20px or smaller): Skip patterns entirely — just use **solid `self.primary` fill** with a `self.comp` stroke. At this scale the *piece silhouette + color* is what identifies the player, not the pattern.

So the factory on `CountryFlag` could look something like:

```python
@patch
def piece_fill(self: CountryFlag, pat_id: str, size: str = 'board') -> tuple:
    """Returns (fill_str, pattern_def_or_None) for the given rendering tier."""
```

Where `'board'` returns `(self.primary, None)` (no def needed), `'list'` picks from a small subset with appropriate scale, and `'large'` uses the full `flagPattern`.

Then `_ensure_pattern` just calls `self.flag.piece_fill(pat_name, size=...)` and only registers the def when it gets one back.

Does that match what you're thinking? Want me to write it out?

### Piece Plan

In [ ]:
#| export
@patch
def piece_fill(self: CountryFlag, pat_id: str, size: str = 'board',
               scale: float = 1.0) -> tuple:
    if size == 'board':
        return self.primary, None

    if size == 'list':
        _SIMPLE = [
            self.bicolorH, self.bicolorV, self.tricolorH,
            self.quarters, self.crossFlag, self.chevron,
        ]
        fn = _SIMPLE[self.patternIndex % len(_SIMPLE)]
        pat = fn(pat_id)
        # Scale tiles to match rendered piece size (~45 unit piece × scale = pixels)
        # This gives ~1 clean tile repetition across the piece
        pat.attributes['patternTransform'] = f'scale({scale:.2f})'
        return f"url(#{pat_id})", pat

    # 'large'
    pat = self.flagPattern(pat_id, scale=0.1)
    return f"url(#{pat_id})", pat


In [ ]:
#| export
@patch
def _piece_renderer(self: CountryFlag, piece_type: PieceType):
    """Return the bound method for drawing a given piece type."""
    return {
        PieceType.KING:   self.kingPiece,
        PieceType.QUEEN:  self.queenPiece,
        PieceType.BISHOP: self.bishopPiece,
        PieceType.KNIGHT: self.knightPiece,
        PieceType.ROOK:   self.rookPiece,
        PieceType.PAWN:   self.pawnPiece,
    }[piece_type]


@patch
def piece_svg(self: CountryFlag, piece_type: PieceType, center: MapCord,
              scale: float = 1.0, size: str = 'board',
              piece_id: str = None, attrs: dict = None) -> tuple:
    """Render a chess piece SVG with appropriate fill for the display tier.

    Returns (svg_string, pattern_def | None).
    Caller registers the def on their SVGBuilder / grid.builder if not None.

    Args:
        piece_type: PieceType enum value
        center:     MapCord placement position
        scale:      SVG scale factor (auto-sized from hex radius when None)
        size:       'board' | 'list' | 'large'
        piece_id:   unique SVG element id  (auto-generated if None)
        attrs:      dict of HTMX / HTML attributes for interactivity
    """
    if piece_id is None:
        piece_id = f"{piece_type.value}_{self.name}"

    pat_id = f"{piece_id}_pat"
    #fill, pat_def = self.piece_fill(pat_id, size=size)
    fill, pat_def = self.piece_fill(pat_id, size=size, scale=scale)
    svg = self._piece_renderer(piece_type)(
        center, scale=scale, piece_id=piece_id, fill=fill, attrs=attrs
    )
    return svg, pat_def


@patch
def draw_piece(self: CountryFlag, piece_type: PieceType, center: MapCord,
               builder: SVGBuilder, scale: float = 1.0, size: str = 'board',
               piece_id: str = None, layer: str = None,
               opacity: float = 1.0, attrs: dict = None) -> str:
    """Convenience: render piece, register pattern on builder, return SVG.

    If layer is given, also calls builder.adjust(layer, svg).
    """
    svg, pat_def = self.piece_svg(
        piece_type, center, scale=scale, size=size,
        piece_id=piece_id, attrs=attrs,
    )
    if pat_def is not None:
        builder.add_definition(pat_def)
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer is not None:
        builder.adjust(layer, svg)
    return svg


can you write it out. I think these functions should take a PieceType so we don't have _PIRECE_RENDERERS

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's the full factory approach. First, let's give `PieceType` an icon property, then put everything on `CountryFlag`:

```python
class PieceType(Enum):
    """Chess piece types."""
    KING   = "king"
    QUEEN  = "queen"
    BISHOP = "bishop"
    KNIGHT = "knight"
    ROOK   = "rook"
    PAWN   = "pawn"

    @property
    def icon(self):
        return {"king": "♚", "queen": "♛", "bishop": "♝",
                "knight": "♞", "rook": "♜", "pawn": "♟"}[self.value]
```

```python
@patch
def piece_fill(self: CountryFlag, pat_id: str, size: str = 'board') -> tuple:
    """Return (fill_string, pattern_def_or_None) for the display tier.

    Tiers:
        'board' — solid primary fill, no pattern def  (≤20px pieces)
        'list'  — simple pattern subset at list scale  (~40px sidebar)
        'large' — full flagPattern dispatch             (gallery/detail)
    """
    if size == 'board':
        return self.primary, None

    if size == 'list':
        _SIMPLE = [
            self.bicolorH, self.bicolorV, self.tricolorH,
            self.quarters, self.crossFlag, self.chevron,
        ]
        fn = _SIMPLE[self.patternIndex % len(_SIMPLE)]
        return f"url(#{pat_id})", fn(pat_id)

    # 'large'
    pat = self.flagPattern(pat_id, scale=0.1)
    return f"url(#{pat_id})", pat
```

```python
@patch
def _piece_renderer(self: CountryFlag, piece_type: PieceType):
    """Return the bound method for drawing a given piece type."""
    return {
        PieceType.KING:   self.kingPiece,
        PieceType.QUEEN:  self.queenPiece,
        PieceType.BISHOP: self.bishopPiece,
        PieceType.KNIGHT: self.knightPiece,
        PieceType.ROOK:   self.rookPiece,
        PieceType.PAWN:   self.pawnPiece,
    }[piece_type]


@patch
def piece_svg(self: CountryFlag, piece_type: PieceType, center: MapCord,
              scale: float = 1.0, size: str = 'board',
              piece_id: str = None, attrs: dict = None) -> tuple:
    """Render a chess piece SVG with appropriate fill for the display tier.

    Returns (svg_string, pattern_def | None).
    Caller registers the def on their SVGBuilder / grid.builder if not None.

    Args:
        piece_type: PieceType enum value
        center:     MapCord placement position
        scale:      SVG scale factor (auto-sized from hex radius when None)
        size:       'board' | 'list' | 'large'
        piece_id:   unique SVG element id  (auto-generated if None)
        attrs:      dict of HTMX / HTML attributes for interactivity
    """
    if piece_id is None:
        piece_id = f"{piece_type.value}_{self.name}"

    pat_id = f"{piece_id}_pat"
    fill, pat_def = self.piece_fill(pat_id, size=size)
    svg = self._piece_renderer(piece_type)(
        center, scale=scale, piece_id=piece_id, fill=fill, attrs=attrs
    )
    return svg, pat_def


@patch
def draw_piece(self: CountryFlag, piece_type: PieceType, center: MapCord,
               builder: SVGBuilder, scale: float = 1.0, size: str = 'board',
               piece_id: str = None, layer: str = None,
               opacity: float = 1.0, attrs: dict = None) -> str:
    """Convenience: render piece, register pattern on builder, return SVG.

    If layer is given, also calls builder.adjust(layer, svg).
    """
    svg, pat_def = self.piece_svg(
        piece_type, center, scale=scale, size=size,
        piece_id=piece_id, attrs=attrs,
    )
    if pat_def is not None:
        builder.add_definition(pat_def)
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer is not None:
        builder.adjust(layer, svg)
    return svg
```

Then your downstream code collapses to:

```python
@patch
def draw_svg(self: Piece, grid: HexGrid, hex_idx: int = None,
             scale: float = None, opacity: float = 1.0,
             suffix: str = "") -> str:
    idx = hex_idx if hex_idx is not None else self.location
    if idx is None or idx < 0 or idx >= len(grid.hexes) or not self.flag:
        return ""

    center = grid.hexes[idx].center
    if scale is None:
        scale = (grid.radius * 0.75) / 22.5

    # Auto-select tier from scale
    size = 'board' if scale < 0.6 else ('list' if scale < 1.5 else 'large')

    return self.flag.draw_piece(
        self.piece_type, center, grid.builder,
        scale=scale, size=size,
        piece_id=f"piece_{self.id[:8]}{suffix}",
        opacity=opacity,
    )


@patch
def draw_ghost(self: Piece, grid: HexGrid, hex_idx: int,
               scale: float = None, opacity: float = 0.4) -> str:
    s = scale or (grid.radius * 0.75) / 22.5
    return self.draw_svg(grid, hex_idx=hex_idx,
                         scale=s * 0.7, opacity=opacity,
                         suffix="_ghost")
```

The key design points:

- **`piece_fill`** — pure data, returns fill + optional def. Knows the three tiers.
- **`piece_svg`** — combines fill + renderer dispatch, returns SVG + def. No side effects.
- **`draw_piece`** — convenience that also registers the def on a builder. The "batteries included" version.
- **`_piece_renderer`** — replaces `_PIECE_RENDERERS` dict. Lives on the flag so PieceType → method is internal.
- **Tier auto-detection** in `draw_svg` uses the computed scale as a heuristic — you can override with explicit `size=` when needed.

The threshold breakpoints (0.6 / 1.5) are tunable — those are just where the patterns start becoming illegible in my experience with your patterns.

Can you build a gallery of the different sizes for the queen and a random country?
    

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.QUEEN, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()
```

This renders the same queen piece at each of the three tiers side-by-side:

- **board** (scale 0.5) — solid `primary` fill, no pattern registered, clean at tiny sizes
- **list** (scale 1.5) — picks from the 6 simple patterns (bicolor, tricolor, quarters, cross, chevron)
- **large** (scale 3.5) — full `flagPattern` dispatch with all 16 options

You can swap `PieceType.QUEEN` for any other piece type and it'll work the same way through `draw_piece`.

In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.QUEEN, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()


There seems to be some wrong colors for the middle one of list at 1.5 is it a scale issue?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, it's a scale issue — two related problems:

1. **Tile size doesn't track render scale.** The simple patterns have fixed ~42px tiles in `userSpaceOnUse` coords. At `list` scale 1.5× the piece renders ~67px, so you see ~4–5 stripes instead of a clean 2–3.
2. **Pattern anchors to SVG origin, not the piece.** Different piece positions show different slices of the same tile, so the colors can look arbitrary.

Fix: pass `scale` into `piece_fill` and apply it as `patternTransform` on the list patterns:

```python
@patch
def piece_fill(self: CountryFlag, pat_id: str, size: str = 'board',
               scale: float = 1.0) -> tuple:
    if size == 'board':
        return self.primary, None

    if size == 'list':
        _SIMPLE = [
            self.bicolorH, self.bicolorV, self.tricolorH,
            self.quarters, self.crossFlag, self.chevron,
        ]
        fn = _SIMPLE[self.patternIndex % len(_SIMPLE)]
        pat = fn(pat_id)
        # Scale tiles to match rendered piece size (~45 unit piece × scale = pixels)
        # This gives ~1 clean tile repetition across the piece
        pat.attributes['patternTransform'] = f'scale({scale:.2f})'
        return f"url(#{pat_id})", pat

    # 'large'
    pat = self.flagPattern(pat_id, scale=0.1)
    return f"url(#{pat_id})", pat
```

Then in `piece_svg`, thread `scale` through:

```python
fill, pat_def = self.piece_fill(pat_id, size=size, scale=scale)
```

With `scale=1.5`, the 42px tile becomes 63px — and the ~67px piece shows exactly 1 clean tile repetition (3 stripes for tricolorH, 4 quads for quarters, etc.). At `board` it doesn't matter since there's no pattern. At `large` the `flagPattern` already handles its own internal scaling.

In [ ]:
#!cat ../../docs/*.md

from dataclasses import dataclass, field
from typing import Optional, List, Set
from enum import Enum
import uuid

class PieceGoal(Enum):
    """Available goals for pieces"""
    EXPLORE = "explore"
    HARVEST = "harvest"
    ATTACK = "attack"
    MOVE = "move"
    SETTLE = "settle"
    DEFEND = "defend"
    PATROL = "patrol"

class PersonalityTrait(Enum):
    """Personality tendencies when acting autonomously"""
    AGGRESSIVE = "aggressive"      # Prefers attack/explore
    DEFENSIVE = "defensive"        # Prefers defend/settle
    ECONOMIC = "economic"          # Prefers harvest/settle
    EXPLORER = "explorer"          # Prefers explore/move
    BALANCED = "balanced"          # No strong preference




@dataclass
class Piece:
    """A game piece representing a group of units."""
    
    # Identity
    id: str = field(default_factory=lambda: str(uuid.uuid4()))
    owner_id: int = 0  # Kingdom/country ID
    parent_id: Optional[str] = None  # ID of piece that spawned this
    
    # Core attributes
    size: int = 100  # Number of units in this piece
    health: int = 100  # Hit points (0-100 scale)
    max_health: int = 100
    
    # Vision and intelligence
    sight: int = 3  # How many hex rings they can see
    memory: float = 0.8  # Retention rate (0-1, higher = better memory)
    
    # Movement
    movement_range: int = 4  # Max weighted hexes per turn
    current_position: int = -1  # Current hex index
    
    # Goals and targeting
    goal: PieceGoal = PieceGoal.EXPLORE
    target_position: Optional[int] = None  # Target hex index
    target_piece: Optional[str] = None  # Target piece UUID
    
    # Behavior
    personality: PersonalityTrait = PersonalityTrait.BALANCED
    
    # Relationships
    spawned_pieces: List[str] = field(default_factory=list)  # UUIDs of children
    
    # Knowledge (what hexes this piece knows about)
    known_hexes: Set[int] = field(default_factory=set)
    knowledge_freshness: dict = field(default_factory=dict)  # hex_idx -> turn_last_seen
    
    # Settlement state
    is_settled: bool = False
    settle_progress: int = 0  # Turns spent settling (settlement complete at threshold)
    settle_threshold: int = 3


In [ ]:
import os
import glob

def concatenate_text_files(directory_path):
    # Change the current working directory to the specified path
    os.chdir(directory_path)
    
    # Use glob to find all files matching a pattern (e.g., all .txt files)
    # Adjust '*.txt' to match your specific file extension if needed
    file_list = glob.glob('*.py') 
    
    # Open the output file in write mode ('w')
    ret = ""
    for fname in file_list:
        print(fname)
        with open(fname, 'r') as infile:
            # Read the content and write it to the output file
            ret += f"{fname}\n\n " + infile.read() 
            # Optional: Add a newline character as a separator between files
            #outfile.write('\n')
    return ret

In [ ]:
primitives = concatenate_text_files("../../HexMagic/plot")

does $`primitives` have information about HexRegions

In [ ]:
rootIdeas  = concatenate_text_files("../../HexMagic")

in $`rootIdeas` do you see how terrain has encode and decode. can we build something similar for piece

In [ ]:
from HexMagic.core import Terrain

In [ ]:
??Terrain.encode

Does the terrain style encoding make sense to work for piece

So we have a bunch of patterns for the flag ( sheridan, weave, yin, swirl, triPattern ) but ultimately the country should just have one pattern. can we build a function that will assign one pattern. This will need to be consistent (ie encode and decode) so maybe it is an int index into a function 

Can you write it?

In [ ]:
#| export
@patch
def flag_banner(self: CountryFlag, width=150, height=100,
                pattern_fn=None, wavy=False, pole=True, attrs=None):
    """Traditional rectangular flag with optional pole and wavy edge.
    
    Returns SVGBuilder. Use .show() for Jupyter or wrap for FastHTML:
        Div(NotStr(flag.flag_banner().xml()), **htmx_attrs)
    
    Args:
        width/height: flag dimensions
        pattern_fn: 'swirl','weave','triPattern','circlePattern','yin','sheridan'
                    (defaults via patternIndex)
        wavy: wavy right edge via MapPath.make_windy
        pole: draw flagpole on left
        attrs: extra SVG attributes dict for the flag polygon
    """
    PATTERNS = ['swirl', 'weave', 'triPattern', 'circlePattern', 'yin', 'sheridan']
    if pattern_fn is None:
        pattern_fn = PATTERNS[self.patternIndex % len(PATTERNS)]
    
    pad, pole_w, finial_r, pole_extra = 6, 5, 6, 30
    
    # Layout offsets
    fx = pad + (pole_w if pole else 0)
    fy = pad + (finial_r * 2 if pole else 0)
    
    builder = SVGBuilder()
    builder.width = fx + width + pad
    builder.height = fy + height + (pole_extra if pole else 0) + pad
    
    # --- Pattern fill ---
    pat_id = f"fp_{self.name}"
    builder.add_definition(getattr(self, pattern_fn)(pat_id))
    flag_style = StyleCSS(f"flag_{self.name}",
                          fill=f"url(#{pat_id})", stroke=self.comp, stroke_width=2.5)
    builder.add_style(flag_style)
    
    # --- Flag shape via MapPath ---
    tl, tr = MapCord(fx, fy), MapCord(fx + width, fy)
    br, bl = MapCord(fx + width, fy + height), MapCord(fx, fy + height)
    
    if wavy:
        right_edge = MapPath([tr, br]).make_windy(iterations=3, offset_factor=0.12)
        points = [tl, tr] + right_edge.points[1:] + [bl]
    else:
        points = [tl, tr, br, bl]
    
    flag_path = MapPath(points, flag_style)
    builder.adjust("flag", flag_path.drawClosed(CountryFlag._render_attrs(attrs)))
    
    # --- Pole + finial ---
    if pole:
        pole_style = StyleCSS(f"pole_{self.name}",
                              stroke=self.darkPrimary, stroke_width=pole_w,
                              stroke_linecap="round", fill=self.darkPrimary)
        builder.add_style(pole_style)
        px = pad + pole_w / 2
        builder.adjust("pole",
            f'<line x1="{px}" y1="{pad + finial_r}" '
            f'x2="{px}" y2="{fy + height + pole_extra}" '
            f'class="{pole_style.name}"/>'
            f'<circle cx="{px}" cy="{pad + finial_r}" '
            f'r="{finial_r}" class="{pole_style.name}"/>')
    
    return builder
